In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cmocean import cm
from scipy.ndimage import gaussian_filter

In [ ]:
zmin = -500
zmax = 0

bat_file = "../bat.xhd"

unit = "m" # m, km, or simply an empty string.

In [ ]:
x = np.loadtxt("X.DAT")
if unit == "m":
    x = x/1e3
z = np.loadtxt("Z.DAT")
final_field = True
try:
    v = np.loadtxt("V_VEL.DAT")
    t = np.loadtxt("TEMP.DAT")
    s = np.loadtxt("SAL.DAT")

    assert len(v) > 0
except:
    final_field = False
    print("Final fields not generated")
vmean = np.loadtxt("Vmean.dat")
tmean = np.loadtxt("Tmean.dat")
smean = np.loadtxt("Smean.dat")
bat = np.loadtxt(bat_file)
diag = np.loadtxt("diagnose.out")
topo_x = bat[:,0]
topo = bat[:,1]

try:
    diag_time = diag[:,0]
    ecin = diag[:,1]
    epot = diag[:,2]
except:
    diag_time = [diag[0]]
    ecin = [diag[1]]
    epot = [diag[2]]
    print("Not enough values to compute")

# Grid Information

In [ ]:
lz, lx = x.shape
dx = np.gradient(x[0])[0]
days_runned = vmean.shape[0]//lz

print(f"GRID REPORT:\ngridpoints in x: {lx}\nσ levels: {lz+1}\nΔx: {dx:.2f} m\nComplete Days: {days_runned:}")

# Energy

In [ ]:
plt.figure(figsize=(15,4))
plt.suptitle(f"Last Time: {diag_time[-1]:.2f} days")
plt.subplot(121)
plt.title("Mean Kinetic Energy")
plt.plot(diag_time, ecin, '-r')
ylim = plt.ylim()
plt.plot([5,5],ylim, '--k')
plt.plot([10,10],ylim, '--k')
plt.plot([15,15],ylim, '--k')
plt.ylim(*ylim)
plt.xlim(diag_time[0], diag_time[-1])
plt.xlabel("Execution Time (days)")
plt.ylabel("MKE")
plt.grid()

plt.subplot(122)
plt.title("Mean Potential Energy")
plt.plot(diag_time, epot, '-r')
ylim = plt.ylim()
plt.plot([5,5],ylim, '--k')
plt.plot([10,10],ylim, '--k')
plt.plot([15,15],ylim, '--k')
plt.ylim(*ylim)
plt.xlim(diag_time[0], diag_time[-1])
plt.xlabel("Execution Time (days)")
plt.ylabel("MPE")
plt.grid()
plt.show()

# Mean Fields

In [ ]:
vlas = vmean[-100:,:]
tlas = tmean[-100:,:]
slas = smean[-100:,:]

In [ ]:
vref = np.nanmax(np.abs(vlas))

plt.figure(figsize=(8,5))
plt.title("Velocidade Meridional - Média do Último Dia - DEPROAS II")
plt.contourf(x, z, vlas, levels=np.linspace(-vref, vref, 31), cmap=cm.balance_r, extend='both')
plt.colorbar(label="Velocity (m s⁻¹)")
c = plt.contour(x, z, v, levels=np.arange(-.5, .6, .25), colors='k', linewidths=1)
plt.clabel(c, c.levels, fmt="%.2f")
plt.fill_between(topo_x, [zmin]*len(topo_x), -topo, color='k')
plt.xlim(0,np.max(x))
plt.ylim(zmin,zmax)
plt.text(20, -290, "v$_\\text{max}$:  "+f"{np.max(vlas):.2f}"+"$\\,$m s$^{-1}$", color='w')
plt.text(20, -315, "v$_\\text{min}$ : "+f"{np.min(vlas):.2f}"+"$\\,$m s$^{-1}$", color='w')
plt.show()

In [ ]:
tmin, tmax = np.nanmin(tlas), np.nanmax(tlas)

plt.figure(figsize=(8,5))
if tmin == tmax:
    plt.contourf(x, z, tlas, cmap=cm.thermal, levels=31, extend='both')
else:
    plt.contourf(x, z, tlas, cmap=cm.thermal, levels=np.linspace(tmin, tmax, 31), extend='both')
plt.colorbar(label="Temperature (ᵒ C)")
plt.fill_between(topo_x, [zmin]*len(topo_x), -topo, color='k')
plt.xlim(0,np.max(x))
plt.ylim(zmin,zmax)
plt.text(20, -290, "T$_\\text{max}$:  "+f"{np.max(tlas):.2f}"+"$^\\circ$ C", color='w')
plt.text(20, -315, "T$_\\text{min}$ : "+f"  {np.min(tlas):.2f}"+"$^\\circ$ C", color='w')
plt.show()

In [ ]:
smin, smax = np.nanmin(slas), np.nanmax(slas)

plt.figure(figsize=(8,5))
if smin == smax:
    plt.contourf(x, z, slas, cmap=cm.haline, levels=31, extend='both')
else:
    plt.contourf(x, z, slas, cmap=cm.haline, levels=np.linspace(smin, smax, 31), extend='both')
plt.colorbar(label="Salinity")
plt.fill_between(topo_x, [zmin]*len(topo_x), -topo, color='k')
plt.xlim(0,np.max(x))
plt.ylim(zmin, zmax)
plt.text(20, -290, "S$_\\text{max}$:  "+f"{np.max(slas):.2f}", color='w')
plt.text(20, -315, "S$_\\text{min}$ : "+f" {np.min(slas):.2f}", color='w')
plt.show()

# Final Fields

In [ ]:
if final_field:
    vref = np.nanmax(np.abs(v))

    plt.figure(figsize=(8,5))
    plt.title("Velocidade Meridional Final - DEPROAS II")
    plt.contourf(x, z, v, levels=np.linspace(-.5, .5, 31), cmap=cm.balance_r, extend='both')
    plt.colorbar(label="Velocity (m s⁻¹)", ticks=np.arange(-.5, .6, .1))
    c = plt.contour(x, z, v, levels=np.arange(-.5, .6, .25), colors='k', linewidths=1)
    plt.clabel(c, c.levels, fmt="%.2f")
    plt.fill_between(topo_x, [zmin]*len(topo_x), -topo, color='k')
    plt.xlim(0,np.max(x))
    plt.ylim(zmin, zmax)
    plt.text(20, -290, "v$_\\text{max}$:  "+f"{np.max(v):.2f}"+"$\\,$m s$^{-1}$", color='w')
    plt.text(20, -315, "v$_\\text{min}$ : "+f"{np.min(v):.2f}"+"$\\,$m s$^{-1}$", color='w')
    plt.show()

In [ ]:
if final_field:
    sig = 8
    v_fil = gaussian_filter(v, sigma=sig)

    vref = np.nanmax(np.abs(v_fil))

    plt.figure(figsize=(8,5))
    plt.title("Velocidade Meridional Filtrada - DEPROAS II")
    plt.contourf(x, z, v_fil, levels=np.linspace(-.5, .5, 31), cmap=cm.balance_r, extend='both')
    plt.colorbar(label="Velocity (m s⁻¹)", ticks=np.arange(-.5, .6, .1))
    c = plt.contour(x, z, v_fil, levels=np.arange(-.5, .6, .25), colors='k', linewidths=1)
    plt.clabel(c, c.levels, fmt="%.2f")
    plt.fill_between(topo_x, [zmin]*len(topo_x), -topo, color='k')
    plt.xlim(0,np.max(x))
    plt.ylim(zmin, zmax)
    plt.text(35, -265, "$\\sigma$ = "+f"{sig}", color='w')
    plt.text(20, -290, "v$_\\text{max}$:  "+f"{np.max(v_fil):.2f}"+"$\\,$m s$^{-1}$", color='w')
    plt.text(20, -315, "v$_\\text{min}$ : "+f"{np.min(v_fil):.2f}"+"$\\,$m s$^{-1}$", color='w')
    plt.show()

In [ ]:
if final_field:
    tmin, tmax = np.nanmin(t), np.nanmax(t)

    plt.figure(figsize=(8,5))
    if tmin == tmax:
        plt.contourf(x, z, t, cmap=cm.thermal, levels=31, extend='both')
    else:
        plt.contourf(x, z, t, cmap=cm.thermal, levels=np.linspace(tmin, tmax, 31), extend='both')
    plt.colorbar(label="Temperature (ᵒ C)")
    plt.fill_between(topo_x, [zmin]*len(topo_x), -topo, color='k')
    plt.xlim(0,np.max(x))
    plt.ylim(zmin, zmax)
    plt.text(20, -290, "T$_\\text{max}$:  "+f"{np.max(t):.2f}"+"$^\\circ$ C", color='w')
    plt.text(20, -315, "T$_\\text{min}$ : "+f"  {np.min(t):.2f}"+"$^\\circ$ C", color='w')
    plt.show()

In [ ]:
if final_field:
    smin, smax = np.nanmin(s), np.nanmax(s)

    plt.figure(figsize=(8,5))
    if smin == smax:
        plt.contourf(x, z, s, cmap=cm.haline, levels=31, extend='both')
    else:
        plt.contourf(x, z, s, cmap=cm.haline, levels=np.linspace(smin, smax, 31), extend='both')
    plt.colorbar(label="Salinity")
    plt.fill_between(topo_x, [zmin]*len(topo_x), -topo, color='k')
    plt.xlim(0,np.max(x))
    plt.ylim(zmin, zmax)
    plt.text(20, -290, "S$_\\text{max}$:  "+f"{np.max(s):.2f}", color='w')
    plt.text(20, -315, "S$_\\text{min}$ : "+f" {np.min(s):.2f}", color='w')
    plt.show()